# RAG Exercise Solution

In [4]:
import dotenv

from agents import Agent, Runner, trace, function_tool

import chromadb

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [5]:
# We populated the RAG with the data from the data/calories.csv file in the rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient(path="../chroma")
calories_db = chroma_client.get_collection(name="nutrition_db")
nuntrition_qna_db = chroma_client.get_collection(name="nutrition_qna")

In [6]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function too look up calorie information.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = calories_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)



In [7]:

@function_tool
def nutrtition_qna_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function too ask a question about nutrition.

    Args:
        query: The question to ask
        max_results: The maximum number of results to return.

    Returns:
        A string containing the question and the answer related to the query.
    """

    results = nuntrition_qna_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(doc)

    return "Related answers to your question:\n" + "\n".join(formatted_results)

In [8]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information and nutrtion advice    .
    You give concise answers.
    
    If you need to look up calorie information, use the calorie_lookup_tool.
    If are asked a question about nutrition, always use the nutrtition_qna_tool first to see if there is an answer in the knowledge base.
    """,
    tools=[calorie_lookup_tool, nutrtition_qna_tool],
)

In [9]:
with trace("Nutrition Assistant with Nutrition and Calorie RAG"):
    result = await Runner.run(
        calorie_agent,
        "What are the best meal choices for pregnant women and how many calories do they have?",
    )
    print(result.final_output)

Here are general, nutrition-focused meal ideas for pregnancy with approximate calories. Exact needs vary by person and trimester.

- Grilled salmon, quinoa, and steamed broccoli: about 450–550 kcal
  - Source of protein, DHA, iron, and calcium (if you add a cheese or yogurt side).

- Lentil curry with brown rice and spinach: about 400–500 kcal
  - Plant protein, fiber, iron, folate.

- Greek yogurt parfait with berries, granola, and chia seeds: about 300–420 kcal
  - Protein, calcium, probiotics, fiber.

- Veggie omelet with spinach, tomatoes, feta, and whole‑grain toast: about 350–450 kcal
  - Protein, folate, iron, complex carbs.

- Chicken avocado salad with chickpeas and olive oil/vinegar dressing: about 500–600 kcal
  - Lean protein, healthy fats, fiber, iron.

Tips for pregnant meals
- Aim for regular meals with a protein source at each (20–30 g) and plenty of fruits/vegetables.
- Include iron-rich foods (lean meats, beans, fortified cereals) and vitamin C to aid absorption.
- In

In [10]:
with trace("Nutrition Fruit ssistant with Nutrition and Calorie RAG"):
    result = await Runner.run(
        calorie_agent,
        "What are the top 5 fruit choices for health benefit and how many calories do they have?",
    )
    print(result.final_output)

Top 5 fruit choices for health (common, nutrient-rich picks) and calories per 100 g:

- Strawberries: 32 kcal
- Blueberries: 57 kcal
- Apple: 52 kcal
- Orange: 47 kcal
- Banana: 89 kcal

Notes:
- All are high in fiber, vitamins (notably vitamin C), and antioxidants.
- Calories vary with size and preparation (e.g., dried, juiced).
